### Train neural network to evaluate Big G project 

### Model test of interest: Neural Network
* Apply neural network and evaluate using kfold cross validation
* Correct for imbalanced dataset
* After performance evaluation, explore feature impact and evaluation

In [29]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier, MLPRegressor

from sklearn.preprocessing import StandardScaler, OneHotEncoder, MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

from sklearn.inspection import PartialDependenceDisplay

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer

import warnings
warnings.filterwarnings("ignore")

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [2]:
# Eventually, we want to define a way to evaluate the amount of money saved by the model
# This will help us compare different models better 
def calculate_cost_savings(y_true, y_pred):
    """
    Returns the amount of money we are saving Big G by implementing our model
    """
    # ravel() extracts values in order: TN, FP, FN, TP
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    # Calculate weighted cost
    cost = (tp * 4000) -(fp * 500) -(fn * 4000) 
    
    return cost

### Define feature and target columns

In [100]:
## target columns
target_var = ['derate_2_6_hr',
              'derate_6_12_hr',
              'derate_12_24_hr']

## Predictor columns
predict_var= ['year',
           'month',
           'weekday',
              'hour',
              'DistanceLtd',
              'EngineTimeLtd',
              'FuelLtd',
              'activeTransitionCount',
              'TurboBoostPressure',
              'AcceleratorPedal',
              'BarometricPressure',
              'CruiseControlSetSpeed',
              'EngineCoolantTemperature',
              'EngineLoad',
              'EngineOilPressure',
              'EngineOilTemperature',
              'EngineRpm',
              'FuelLevel',
              'FuelRate',
              'FuelTemperature',
              'IntakeManifoldTemperature',
              'Speed',
              'SwitchedBatteryVoltage',
              'Throttle',
              'active',
              'CruiseControlActive',
              'IgnStatus',
              'ParkingBrake',
              'spn',
              'fmi',
              'min_since_last_fault',
              'EquipmentID',
              'eventDescription',
              'LampStatus']

### Apply model
* source for SMOTE rebalancing: source: https://machinelearningmastery.com/smote-oversampling-for-imbalanced-classification/

In [105]:
def apply_NN_classifier(train_data, val_data, target_var, over=True, overunder=False):
    global nn_pipe
    #define target and feature list 
    target_var= target_var
    print("Model for ", target_var)
    
    #Prepare train and test variables from the datasets 
    X_train = train_data.drop(columns=target_var)
    y_train= train_data[target_var].astype(int)
    X_test = val_data.drop(columns=target_var)
    y_test=val_data[target_var].astype(int)

    #Use SMOTE to over sample the minroity class and under sample the majority class 
    #print("before imbalance correction", y_train.value_counts())
    
    # Apply SMOTE only to training--> OVERSAMPLE minority class
    smote = SMOTE(random_state=321, sampling_strategy=0.1)
    X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
    #print("oversample counts", y_resampled.value_counts())

    # Apply SMOTE only to training--> UNDERSAMPLE majority class
    under_sample= RandomUnderSampler(random_state=321, sampling_strategy=0.5)
    X_undersample, y_undersample = under_sample.fit_resample(X_resampled, y_resampled)
    #print("undersample counts", y_undersample.value_counts())

    #Neural network model application and view classification report
    nn_pipe = Pipeline(
    steps = [
        ('model', MLPClassifier(random_state=321, 
                                max_iter=5000, 
                                hidden_layer_sizes = (100,100), 
                                activation='relu', 
                               early_stopping=True, 
                               solver='adam'))
    ])

    if over==True:
        X_train= X_resampled
        y_train= y_resampled
        nn_pipe.fit(X_train, y_train)
        y_pred = nn_pipe.predict(X_test)
        
        print("Trained to over sample")
        #print(classification_report(y_test, y_pred))
    elif overunder==True:
        X_train= X_undersample
        y_train= y_undersample
        nn_pipe.fit(X_train, y_train)
        y_pred = nn_pipe.predict(X_test)
        print("Trained to over and under sample")
        #print(classification_report(y_test, y_pred))
    else:
        nn_pipe.fit(X_train, y_train)
        y_pred = nn_pipe.predict(X_test)
        print("Not corrected for imbalanced")
        #print(classification_report(y_test, y_pred))

    #Show classification report 
    test_results= classification_report(y_test, y_pred, output_dict=True)

    #Calcualte total cost 
    total_cost = calculate_cost_savings(y_test, y_pred)

    return target_var, test_results, total_cost

### Loop through all the datasets and apply the model to each, and save results in final dataframe for model evaluation decisions

In [108]:
model_evaluation_results= pd.DataFrame(columns=['time_window', 'fold_num', 'oversample', 'over-undersample', 'no-correction', 'f1-score', 'total_cost'])
k_fold_num= [1, 3, 4]
##iteration 1: open each csv file for each fold 
for fold in k_fold_num:
    fold_num = fold
    print(f"Applying k-fold number {fold}")
    
    #open train dataset csv file
    train_data= pd.read_csv(f'../processed_data/train_set_{fold}.csv')
    #open validation dataset csv file 
    val_data= pd.read_csv(f'../processed_data/val_set_{fold}.csv')

    ##iteration 2: each target variable
    for var in range(len(target_var)):
        target_var_opt= target_var[var]
        print(target_var_opt)
        #Run model with over-sample 
        test_target, classrep, total_cost= apply_NN_classifier(train_data, val_data, target_var_opt, over=True, overunder=False)
        model_evaluation_results.loc[len(model_evaluation_results)]= [target_var_opt, fold, 1, 0, 0, classrep['macro avg']['f1-score'], total_cost]
    
        #Run model with over and under sample 
        test_target, classrep, total_cost= apply_NN_classifier(train_data, val_data, target_var_opt, over=False, overunder=True)
        model_evaluation_results.loc[len(model_evaluation_results)]= [target_var_opt, fold, 0, 1, 0, classrep['macro avg']['f1-score'], total_cost]
    
        #Run model with no imbalance correction
        test_target, classrep, total_cost= apply_NN_classifier(train_data, val_data, target_var_opt, over=False, overunder=False)
        model_evaluation_results.loc[len(model_evaluation_results)]= [target_var_opt, fold, 0, 0, 1, classrep['macro avg']['f1-score'], total_cost]

#view final dataframe 
model_evaluation_results

Applying k-fold number 1
derate_2_6_hr
Model for  derate_2_6_hr
Trained to over sample
Model for  derate_2_6_hr
Trained to over and under sample
Model for  derate_2_6_hr
Not corrected for imbalanced
derate_6_12_hr
Model for  derate_6_12_hr
Trained to over sample
Model for  derate_6_12_hr
Trained to over and under sample
Model for  derate_6_12_hr
Not corrected for imbalanced
derate_12_24_hr
Model for  derate_12_24_hr
Trained to over sample
Model for  derate_12_24_hr
Trained to over and under sample
Model for  derate_12_24_hr
Not corrected for imbalanced
Applying k-fold number 3
derate_2_6_hr
Model for  derate_2_6_hr
Trained to over sample
Model for  derate_2_6_hr
Trained to over and under sample
Model for  derate_2_6_hr
Not corrected for imbalanced
derate_6_12_hr
Model for  derate_6_12_hr
Trained to over sample
Model for  derate_6_12_hr
Trained to over and under sample
Model for  derate_6_12_hr
Not corrected for imbalanced
derate_12_24_hr
Model for  derate_12_24_hr
Trained to over sampl

,time_window,fold_num,oversample,over-undersample,no-correction,f1-score,total_cost
0,derate_2_6_hr,1,1,0,0,0.498418,-1204500
1,derate_2_6_hr,1,0,1,0,0.492808,-3813000
2,derate_2_6_hr,1,0,0,1,0.499769,-696000
3,derate_6_12_hr,1,1,0,0,0.498624,-1267000
4,derate_6_12_hr,1,0,1,0,0.487065,-5442000
5,derate_6_12_hr,1,0,0,1,0.499807,-584000
6,derate_12_24_hr,1,1,0,0,0.498215,-1931500
7,derate_12_24_hr,1,0,1,0,0.505970,-1579000
8,derate_12_24_hr,1,0,0,1,0.499523,-1440000
9,derate_2_6_hr,3,1,0,0,0.499249,-762500


In [110]:
model_evaluation_results.sort_values(by='total_cost')

,time_window,fold_num,oversample,over-undersample,no-correction,f1-score,total_cost
19,derate_2_6_hr,4,0,1,0,0.182168,-73108000
22,derate_6_12_hr,4,0,1,0,0.372098,-38571500
25,derate_12_24_hr,4,0,1,0,0.402445,-31130000
13,derate_6_12_hr,3,0,1,0,0.422563,-25636000
10,derate_2_6_hr,3,0,1,0,0.432097,-22974000
4,derate_6_12_hr,1,0,1,0,0.487065,-5442000
1,derate_2_6_hr,1,0,1,0,0.492808,-3813000
16,derate_12_24_hr,3,0,1,0,0.495726,-2283500
6,derate_12_24_hr,1,1,0,0,0.498215,-1931500
7,derate_12_24_hr,1,0,1,0,0.505970,-1579000


In [114]:
model_evaluation_results.groupby(['oversample','over-undersample', 'no-correction', 'time_window'])['f1-score'].mean().sort_values()

oversample  over-undersample  no-correction  time_window    
0           1                 0              derate_2_6_hr      0.369024
                                             derate_6_12_hr     0.427242
                                             derate_12_24_hr    0.468047
1           0                 0              derate_12_24_hr    0.499014
                                             derate_2_6_hr      0.499130
                                             derate_6_12_hr     0.499423
0           0                 1              derate_12_24_hr    0.499730
                                             derate_2_6_hr      0.499822
                                             derate_6_12_hr     0.499830
Name: f1-score, dtype: float64

In [115]:
model_evaluation_results.to_csv('../processed_data/neural-network model performance results.csv')

In [119]:
model_evaluation_results['f1-score'].max()

0.505969551762485

----

### evaluate from unseen 2019 data

In [154]:
data_2019= pd.read_csv(r"C:\Users\Mullo\Documents\NSS_Projects\big-g-highway-to-ml\processed_data\2019-final-testdata_imputed.csv")
data_2019.head(5)

,Unnamed: 0,min_since_last_fault,year,DistanceLtd,EngineTimeLtd,FuelLtd,activeTransitionCount,TurboBoostPressure,AcceleratorPedal,BarometricPressure,...,month_cos__month,weekday_sin__weekday,weekday_cos__weekday,hour_sin__hour,hour_cos__hour,EquipmentID,EventTimeStamp,derate_2_6_hr,derate_6_12_hr,derate_12_24_hr
0,0,0.000234,0.999505,0.229315,0.765025,0.759574,0.188976,0.015625,0.000000,0.956522,...,0.866025,0.781831,0.62349,0.000000,1.000000,1746,2019-01-01 00:24:10,0,0,0
1,1,0.000001,0.999505,0.229315,0.765025,0.759574,0.188976,0.127515,0.242783,0.952519,...,0.866025,0.781831,0.62349,0.000000,1.000000,1746,2019-01-01 00:36:08,0,0,0
2,2,0.000122,0.999505,0.241739,0.706029,0.791824,0.055118,0.000000,0.000000,0.932367,...,0.866025,0.781831,0.62349,0.258819,0.965926,1777,2019-01-01 01:54:35,0,0,0
3,3,0.000001,0.999505,0.241739,0.706029,0.791824,0.055118,0.127988,0.234877,0.952525,...,0.866025,0.781831,0.62349,0.500000,0.866025,1777,2019-01-01 02:08:21,0,0,0
4,4,0.000187,0.999505,0.235307,0.901709,0.782407,0.992126,0.460938,0.984252,0.946860,...,0.866025,0.781831,0.62349,0.707107,0.707107,1741,2019-01-01 03:06:52,0,0,0


### open the imputed data befor 2019 as final test dataset 

In [155]:
full_faults= pd.read_csv(r"C:\Users\Mullo\Documents\NSS_Projects\big-g-highway-to-ml\processed_data\final full train data.csv")

### prepare test and train dataset 

In [158]:
drop_col_1 = target_var + ['EquipmentID','EventTimeStamp']
X_train= full_faults.drop(columns =  target_var)
y_train=full_faults[target_var].astype(int)
y_train= y_train[target_var[1]]

X_test = data_2019.drop(columns = drop_col_1)
y_test=data_2019[target_var].astype(int)
y_test= y_test[target_var[1]]

In [160]:

final_evaluation_results= pd.DataFrame(columns=['time_window', 'oversample', 'over-undersample', 'no-correction', 'f1-score', 'total_cost'])
target_var_opt= target_var[1]
print(target_var_opt)
#Run model with over-sample 
test_target, classrep, total_cost= apply_NN_classifier(train_data, val_data, target_var_opt, over=True, overunder=False)
final_evaluation_results.loc[len(final_evaluation_results)]= [target_var_opt, 1, 0, 0, classrep['macro avg']['f1-score'], total_cost]

#Run model with over and under sample 
test_target, classrep, total_cost= apply_NN_classifier(train_data, val_data, target_var_opt, over=False, overunder=True)
final_evaluation_results.loc[len(final_evaluation_results)]= [target_var_opt, 0, 1, 0, classrep['macro avg']['f1-score'], total_cost]

#Run model with no imbalance correction
test_target, classrep, total_cost= apply_NN_classifier(train_data, val_data, target_var_opt, over=False, overunder=False)
final_evaluation_results.loc[len(final_evaluation_results)]= [target_var_opt,  0, 0, 1, classrep['macro avg']['f1-score'], total_cost]

#view final dataframe 
final_evaluation_results

derate_6_12_hr
Model for  derate_6_12_hr
Trained to over sample
Model for  derate_6_12_hr
Trained to over and under sample
Model for  derate_6_12_hr
Not corrected for imbalanced


,time_window,oversample,over-undersample,no-correction,f1-score,total_cost
0,derate_6_12_hr,1,0,0,0.499862,-416000
1,derate_6_12_hr,0,1,0,0.372098,-38571500
2,derate_6_12_hr,0,0,1,0.499862,-416000


In [161]:
classrep

{'0': {'precision': 0.9994490649997352,
  'recall': 1.0,
  'f1-score': 0.999724456596615,
  'support': 188666.0},
 '1': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 104.0},
 'accuracy': 0.9994490649997352,
 'macro avg': {'precision': 0.4997245324998676,
  'recall': 0.5,
  'f1-score': 0.4998622282983075,
  'support': 188770.0},
 'weighted avg': {'precision': 0.9988984335288449,
  'recall': 0.9994490649997352,
  'f1-score': 0.9991736734028552,
  'support': 188770.0}}

---